# Sentinel-1 water segmentation: a three-day pilot study

**Question:** Does using both VV and VH radar channels improve surface-water segmentation compared with either channel alone?

This notebook downloads the official Sen1Floods11 hand-labelled train/validation/test files, validates them, calculates training-only normalization, trains small U-Nets, and exports a controlled comparison. It replaces the older FCN example with a compact experiment pipeline; it is not an exact reproduction of that model.

**Run:** Open in Colab or Kaggle, enable a GPU and internet access, then run cells in order. No Google account authentication or `gsutil` is required to read these public objects. Defaults: three configurations, one seed, 15 epochs each. Runtime depends on the allocated GPU; measure the first epoch before increasing the budget. Set `SEEDS = [42, 43, 44]` if time permits. CPU supports debugging but is not recommended for the full study.

**Scientific scope:** Labels are water/non-water, including permanent water, not newly flooded/non-flooded land. The official standard splits are used as supplied; they do not establish generalization to an entirely unseen event. This is a replication/exploratory study, not a claim of methodological novelty. The additional geographic holdout is outside this notebook's scope.

References: [dataset and documentation](https://github.com/cloudtostreet/Sen1Floods11), [dataset paper](https://openaccess.thecvf.com/content_CVPRW_2020/html/w11/Bonafilia_Sen1Floods11_A_Georeferenced_Dataset_to_Train_and_Test_Deep_Learning_CVPRW_2020_paper.html). Cite Bonafilia et al., CVPR Workshops 2020, and credit any adapted code in your report.


In [ ]:
%pip -q install rasterio requests pandas matplotlib tqdm


In [ ]:
import os, json, csv, time, random, hashlib, platform, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import rasterio
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

ROOT = Path('/kaggle/working/sen1_study' if Path('/kaggle/working').exists()
            else '/content/sen1_study' if Path('/content').exists() else './sen1_study')
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'
for p in [DATA/'S1', DATA/'Labels', DATA/'splits', RUNS]:
    p.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {'VV': [0], 'VH': [1], 'VV_VH': [0, 1]}
SEEDS = [42]
EPOCHS = 15
BATCH_SIZE = 8
CROP_SIZE = 256
BASE_CHANNELS = 16
LR = 1e-3
NUM_WORKERS = 0  # portable and reproducible across notebooks
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
CLIP_MIN, CLIP_MAX = -50.0, 1.0  # dB; same clipping interval as the supplied example
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('Device:', DEVICE, torch.cuda.get_device_name(0) if AMP else '')
print('Working directory:', ROOT)
print('PyTorch:', torch.__version__, '| Rasterio:', rasterio.__version__)
if not AMP:
    print('Enable a GPU before the full experiment.')


## 1. Download the official files
Only image/label pairs referenced by the three standard CSV splits are downloaded. Files are cached; rerunning does not download successful files again. Downloads use temporary files, retries, and fail loudly. A separate geographic holdout, if present in the dataset, is not silently mixed into these splits.


In [ ]:
BASE_URL = 'https://storage.googleapis.com/sen1floods11/v1.1'
def download(url, target, attempts=4):
    target = Path(target)
    if target.exists() and target.stat().st_size > 0:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + '.part')
    for attempt in range(attempts):
        try:
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with temporary.open('wb') as f:
                    for chunk in response.iter_content(1024*1024):
                        if chunk:
                            f.write(chunk)
            if temporary.stat().st_size == 0:
                raise ValueError('Empty download')
            temporary.replace(target)
            return target
        except (requests.RequestException, OSError, ValueError) as error:
            temporary.unlink(missing_ok=True)
            if attempt == attempts-1:
                raise RuntimeError(f'Download failed: {url}. Check notebook internet access.') from error
            time.sleep(2**attempt)

def read_pairs(path):
    pairs = []
    with Path(path).open(newline='', encoding='utf-8-sig') as f:
        for line_number, row in enumerate(csv.reader(f), 1):
            if not row or all(not x.strip() for x in row):
                continue
            if len(row) != 2 or not all(x.strip().lower().endswith('.tif') for x in row):
                raise ValueError(f'Unexpected CSV row {path}:{line_number}: {row}')
            # CSV entries may contain folder prefixes; files are stored by basename.
            image, label = [x.strip().replace('\\', '/').split('/')[-1] for x in row]
            pairs.append((image, label))
    if not pairs or len(set(pairs)) != len(pairs):
        raise ValueError(f'Empty or duplicate split: {path}')
    return pairs

splits = {}
for split, suffix in [('train','train'), ('valid','valid'), ('test','test')]:
    filename = f'flood_{suffix}_data.csv'
    path = download(f'{BASE_URL}/splits/flood_handlabeled/{filename}', DATA/'splits'/filename)
    splits[split] = read_pairs(path)

for a, b in [('train','valid'), ('train','test'), ('valid','test')]:
    assert not ({x[0] for x in splits[a]} & {x[0] for x in splits[b]}), f'Image overlap: {a}/{b}'
    assert not ({x[1] for x in splits[a]} & {x[1] for x in splits[b]}), f'Label overlap: {a}/{b}'

jobs = {}
for pairs in splits.values():
    for image, label in pairs:
        jobs[DATA/'S1'/image] = f'{BASE_URL}/data/flood_events/HandLabeled/S1Hand/{image}'
        jobs[DATA/'Labels'/label] = f'{BASE_URL}/data/flood_events/HandLabeled/LabelHand/{label}'
with ThreadPoolExecutor(max_workers=4) as pool:
    list(tqdm(pool.map(lambda item: download(item[1], item[0]), jobs.items()),
              total=len(jobs), desc='Downloading/cached files'))
print({split: len(pairs) for split, pairs in splits.items()})


## 2. Validate data and compute training-only statistics
Invalid image pixels (nonfinite values or Rasterio nodata masks) and labels outside the valid water/non-water area are excluded. Label `−1` is mapped directly to ignore value `255` without PIL conversions or scaling. Unexpected classes stop execution.

Normalization uses valid, labelled training pixels after clipping to the documented interval above. Images are normalized in dB units; there is no additional [0,1] scaling. Evaluation uses full tiles; training uses random 256×256 crops and flips. Split assignment is always made before cropping.


In [ ]:
IGNORE = 255
def read_pair(pair):
    image_name, label_name = pair
    with rasterio.open(DATA/'S1'/image_name) as src:
        x = src.read().astype(np.float32)
        image_valid = src.read_masks().all(axis=0) > 0
        image_grid = (src.crs, src.transform, src.width, src.height)
    with rasterio.open(DATA/'Labels'/label_name) as src:
        if src.count != 1:
            raise ValueError(f'Expected one label band: {label_name}')
        y = src.read(1).astype(np.int64)
        label_valid = src.read_masks(1) > 0
        label_grid = (src.crs, src.transform, src.width, src.height)
    if x.shape[0] != 2 or x.shape[1:] != y.shape or image_grid != label_grid:
        raise ValueError(f'Band, shape, or spatial-grid mismatch: {pair}')
    unexpected = set(np.unique(y).tolist()) - {-1, 0, 1, IGNORE}
    if unexpected:
        raise ValueError(f'Unexpected label values {unexpected}: {pair}')
    valid = image_valid & label_valid & np.isfinite(x).all(axis=0) & ((y == 0) | (y == 1))
    y[~valid] = IGNORE
    x = np.clip(np.nan_to_num(x, nan=CLIP_MIN, posinf=CLIP_MAX, neginf=CLIP_MIN), CLIP_MIN, CLIP_MAX)
    return x, y

records = []
channel_sum = np.zeros(2, dtype=np.float64)
channel_sq_sum = np.zeros(2, dtype=np.float64)
pixel_count = 0
for split, pairs in splits.items():
    for pair in tqdm(pairs, desc=f'Checking {split}'):
        x, y = read_pair(pair)
        if min(y.shape) < CROP_SIZE:
            raise ValueError(f'Image smaller than crop: {pair}')
        valid = y != IGNORE
        if not valid.any():
            raise ValueError(f'No valid labelled pixels: {pair}')
        records.append(dict(split=split, image=pair[0], event=pair[0].split('_')[0],
                            valid_pixels=int(valid.sum()), water_pixels=int((y == 1).sum())))
        if split == 'train':
            values = x[:, valid].astype(np.float64)
            channel_sum += values.sum(axis=1)
            channel_sq_sum += (values**2).sum(axis=1)
            pixel_count += values.shape[1]
MEAN = (channel_sum/pixel_count).astype(np.float32)
STD = np.sqrt(np.maximum(channel_sq_sum/pixel_count - MEAN.astype(np.float64)**2, 1e-6)).astype(np.float32)
inventory = pd.DataFrame(records)
inventory.to_csv(ROOT/'data_inventory.csv', index=False)
normalization = dict(mean=MEAN.tolist(), std=STD.tolist(), clip=[CLIP_MIN, CLIP_MAX],
                     source='valid labelled pixels in train split only')
(ROOT/'normalization.json').write_text(json.dumps(normalization, indent=2))
print('VV/VH mean:', MEAN, '| std:', STD)
display(inventory.groupby('split')[['valid_pixels','water_pixels']].sum())
display(pd.crosstab(inventory.event, inventory.split))


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 10))
for row, pair in enumerate(splits['train'][:3]):
    x, y = read_pair(pair)
    for ch in range(2):
        axes[row, ch].imshow(x[ch], cmap='gray', vmin=-30, vmax=0)
        axes[row, ch].set_title(f'{pair[0]} | {"VV" if ch == 0 else "VH"}')
    axes[row, 2].imshow(np.ma.masked_where(y == IGNORE, y), vmin=0, vmax=1, cmap='Blues')
    axes[row, 2].set_title('Water label (white/masked = invalid)')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
fig.savefig(ROOT/'data_preview.png', dpi=140)
plt.show()


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class WaterDataset(Dataset):
    def __init__(self, pairs, channels, train=False, seed=42):
        self.pairs, self.channels, self.train = pairs, channels, train
        # Independent RNG: channel-dependent model initialization cannot change crops.
        self.rng = np.random.default_rng(seed)
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, index):
        x, y = read_pair(self.pairs[index])
        x = (x - MEAN[:, None, None])/STD[:, None, None]
        if self.train:
            h, w = y.shape
            top = int(self.rng.integers(0, h-CROP_SIZE+1))
            left = int(self.rng.integers(0, w-CROP_SIZE+1))
            x = x[:, top:top+CROP_SIZE, left:left+CROP_SIZE]
            y = y[top:top+CROP_SIZE, left:left+CROP_SIZE]
            for axis in [0, 1]:
                if self.rng.random() < 0.5:
                    x = np.flip(x, axis=axis+1)
                    y = np.flip(y, axis=axis)
        return (torch.from_numpy(x[self.channels].copy()).float(),
                torch.from_numpy(y.copy()).long())

def make_loaders(channels, seed):
    generator = torch.Generator().manual_seed(seed)
    train = DataLoader(WaterDataset(splits['train'], channels, True, seed),
                       batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                       num_workers=NUM_WORKERS, pin_memory=AMP)
    valid = DataLoader(WaterDataset(splits['valid'], channels),
                       batch_size=2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=AMP)
    return train, valid


## 3. Small U-Net and checks
All experiments use the same model except the necessary input-channel count. Group normalization supports small batches. Loss is masked binary cross entropy plus soft Dice, calculated in float32. Empty/all-invalid crops are handled without NaNs. Metrics accumulate TP/FP/FN across valid pixels before computing ratios; they are not averages of batch IoUs.


In [ ]:
class Block(nn.Module):
    def __init__(self, inc, outc):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(inc,outc,3,padding=1,bias=False),
            nn.GroupNorm(4,outc), nn.ReLU(inplace=True),
            nn.Conv2d(outc,outc,3,padding=1,bias=False), nn.GroupNorm(4,outc), nn.ReLU(inplace=True))
    def forward(self,x):
        return self.layers(x)

class SmallUNet(nn.Module):
    def __init__(self, in_channels, base=16):
        super().__init__()
        assert base % 4 == 0
        self.enc1, self.enc2 = Block(in_channels,base), Block(base,base*2)
        self.enc3, self.enc4 = Block(base*2,base*4), Block(base*4,base*8)
        self.bridge = Block(base*8,base*16)
        self.dec4, self.dec3 = Block(base*24,base*8), Block(base*12,base*4)
        self.dec2, self.dec1 = Block(base*6,base*2), Block(base*3,base)
        self.out = nn.Conv2d(base,1,1)
    def forward(self,x):
        pool = nn.functional.max_pool2d
        a = self.enc1(x)
        b = self.enc2(pool(a,2))
        c = self.enc3(pool(b,2))
        d = self.enc4(pool(c,2))
        z = self.bridge(pool(d,2))
        for skip, block in [(d,self.dec4),(c,self.dec3),(b,self.dec2),(a,self.dec1)]:
            z = nn.functional.interpolate(z, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            z = block(torch.cat([z,skip],dim=1))
        return self.out(z).squeeze(1)

def segmentation_loss(logits, y):
    valid = y != IGNORE
    if not valid.any():
        return logits.float().sum()*0
    z, target = logits.float()[valid], y[valid].float()
    bce = nn.functional.binary_cross_entropy_with_logits(z,target)
    probabilities = z.sigmoid()
    dice = (2*(probabilities*target).sum()+1)/(probabilities.sum()+target.sum()+1)
    return bce + 1-dice

def confusion_counts(logits, y):
    valid = y != IGNORE
    pred, target = logits >= 0, y == 1  # fixed probability threshold 0.5
    return np.array([int((pred & target & valid).sum()),
                     int((pred & ~target & valid).sum()),
                     int((~pred & target & valid).sum()),
                     int((~pred & ~target & valid).sum())], dtype=np.int64)

def scores(counts):
    tp, fp, fn, tn = map(int, counts)
    def ratio(n,d):
        return n/d if d else float('nan')
    return dict(iou=ratio(tp,tp+fp+fn), precision=ratio(tp,tp+fp), recall=ratio(tp,tp+fn),
                f1=ratio(2*tp,2*tp+fp+fn), tp=tp, fp=fp, fn=fn, tn=tn)

def smoke_checks():
    torch.manual_seed(7)
    model = SmallUNet(2,base=4)
    x = torch.randn(2,2,32,32)
    y = torch.randint(0,2,(2,32,32))
    y[0,:4,:4] = IGNORE
    logits = model(x)
    assert logits.shape == y.shape
    loss = segmentation_loss(logits,y)
    assert torch.isfinite(loss)
    loss.backward()
    changed = logits.detach().clone()
    changed[y == IGNORE] = 1000
    assert torch.allclose(segmentation_loss(logits.detach(),y),segmentation_loss(changed,y))
    assert np.array_equal(confusion_counts(logits,y),confusion_counts(changed,y))
    empty = torch.full_like(y,IGNORE)
    assert segmentation_loss(logits,empty).item() == 0
    toy_logits = torch.tensor([[[2.,2.],[-2.,2.]]])
    toy_labels = torch.tensor([[[1,0],[1,IGNORE]]])
    assert confusion_counts(toy_logits,toy_labels).tolist() == [1,1,1,0]
    assert abs(scores([1,1,1,0])['iou'] - 1/3) < 1e-9
    for channels in [1,2]:
        assert SmallUNet(channels,4)(torch.randn(1,channels,32,32)).shape == (1,32,32)
    print('Passed: model dimensions, backward pass, ignored labels, empty mask, and known metrics.')

smoke_checks()


## 4. Train the three configurations
Use a fixed epoch budget for all models. Save the checkpoint with best validation water IoU. A finished run with an identical configuration is reused; a conflicting saved run raises an error. Interrupted runs restart from epoch one. To deliberately change settings, use a fresh `RUNS` directory in the setup cell. Back up the outputs before a Colab/Kaggle runtime ends.

The default is one seed: this supports a descriptive comparison, not a claim of statistical significance. Three seeds provide a rough measure of run-to-run variability. Crop sequences and sample orders are matched across channel configurations for each seed. Never change settings based on test results.


In [ ]:
DATA_FINGERPRINT = hashlib.sha256(json.dumps(splits,sort_keys=True).encode()).hexdigest()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    counts = np.zeros(4,dtype=np.int64)
    for x,y in loader:
        x,y = x.to(DEVICE),y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            logits = model(x)
        counts += confusion_counts(logits,y)
    return scores(counts)

def run_experiment(name,channels,seed):
    folder = RUNS/f'{name}_seed{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    config = dict(name=name,channels=channels,seed=seed,epochs=EPOCHS,batch_size=BATCH_SIZE,
        crop=CROP_SIZE,base=BASE_CHANNELS,lr=LR,normalization=normalization,
        data_fingerprint=DATA_FINGERPRINT, loss='masked BCE + Dice', threshold=0.5,
        augmentation='crop + hflip + vflip', implementation='small-unet-v1')
    config_path = folder/'config.json'
    if config_path.exists() and json.loads(config_path.read_text()) != config:
        raise ValueError(f'Configuration changed for {folder}; choose a fresh RUNS directory.')
    config_path.write_text(json.dumps(config,indent=2))
    if (folder/'complete.json').exists():
        if not (folder/'best.pt').exists():
            raise FileNotFoundError(f'Missing checkpoint: {folder}')
        print('Reusing completed run:',folder.name)
        return
    seed_everything(seed)
    model = SmallUNet(len(channels),BASE_CHANNELS).to(DEVICE)
    train_loader,valid_loader = make_loaders(channels,seed)
    optimizer = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda',enabled=AMP)
    history, best = [], -1.0
    start = time.time()
    for epoch in range(1,EPOCHS+1):
        model.train()
        losses = []
        for x,y in tqdm(train_loader,desc=f'{name} seed {seed}, epoch {epoch}/{EPOCHS}',leave=False):
            x,y = x.to(DEVICE),y.to(DEVICE)
            if not (y != IGNORE).any():
                continue
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                logits = model(x)
            loss = segmentation_loss(logits,y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Nonfinite loss in {folder}')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
        if not losses:
            raise RuntimeError('No valid training batches.')
        metrics = evaluate(model,valid_loader)
        if not np.isfinite(metrics['iou']):
            raise ValueError('Undefined validation IoU; inspect labels.')
        history.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),
                            valid_iou=metrics['iou'],valid_f1=metrics['f1']))
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        if metrics['iou'] > best:
            best = metrics['iou']
            torch.save(model.state_dict(),folder/'best.pt')
            (folder/'best_validation.json').write_text(json.dumps(dict(epoch=epoch,**metrics),indent=2))
        print(f"{folder.name}: epoch {epoch}, loss {np.mean(losses):.4f}, val IoU {metrics['iou']:.4f}")
    (folder/'complete.json').write_text(json.dumps(dict(seconds=time.time()-start,best_valid_iou=best),indent=2))
    del model,optimizer,scaler
    if AMP:
        torch.cuda.empty_cache()

for name,channels in EXPERIMENTS.items():
    for seed in SEEDS:
        run_experiment(name,channels,seed)


In [ ]:
fig,axes = plt.subplots(1,2,figsize=(11,4))
for name in EXPERIMENTS:
    for seed in SEEDS:
        history = pd.read_csv(RUNS/f'{name}_seed{seed}'/'history.csv')
        axes[0].plot(history.epoch,history.train_loss,label=f'{name}/{seed}')
        axes[1].plot(history.epoch,history.valid_iou,label=f'{name}/{seed}')
axes[0].set(ylabel='Training BCE + Dice loss',xlabel='Epoch')
axes[1].set(ylabel='Validation water IoU',xlabel='Epoch')
for ax in axes:
    ax.legend()
plt.tight_layout()
fig.savefig(ROOT/'training_curves.png',dpi=160)
plt.show()


## 5. Final test evaluation — run after freezing the experiment
Run this section once the training configuration and analysis plan are fixed. Do not use test scores or images to select hyperparameters. Test metrics are pooled over all valid pixels; per-event results below are descriptive because events may also appear in training. Fixed threshold: 0.5.


In [ ]:
def load_model(name,channels,seed):
    model = SmallUNet(len(channels),BASE_CHANNELS).to(DEVICE)
    model.load_state_dict(torch.load(RUNS/f'{name}_seed{seed}'/'best.pt',map_location=DEVICE,weights_only=True))
    return model.eval()

rows, event_rows = [], []
for name,channels in EXPERIMENTS.items():
    for seed in SEEDS:
        model = load_model(name,channels,seed)
        test_loader = DataLoader(WaterDataset(splits['test'],channels),batch_size=2,shuffle=False)
        metrics = evaluate(model,test_loader)
        valid_info = json.loads((RUNS/f'{name}_seed{seed}'/'best_validation.json').read_text())
        rows.append(dict(experiment=name,seed=seed,best_epoch=valid_info['epoch'],**metrics))
        for event in sorted({p[0].split('_')[0] for p in splits['test']}):
            pairs = [p for p in splits['test'] if p[0].split('_')[0] == event]
            loader = DataLoader(WaterDataset(pairs,channels),batch_size=2,shuffle=False)
            event_rows.append(dict(experiment=name,seed=seed,event=event,n_tiles=len(pairs),**evaluate(model,loader)))
        del model
results = pd.DataFrame(rows)
results.to_csv(ROOT/'test_results.csv',index=False)
pd.DataFrame(event_rows).to_csv(ROOT/'test_results_by_event.csv',index=False)
display(results[['experiment','seed','best_epoch','iou','precision','recall','f1']])
summary = results.groupby('experiment')[['iou','precision','recall','f1']].agg(['mean','std'])
summary.to_csv(ROOT/'test_summary.csv')
display(summary)
print('NaN standard deviation is expected with one seed. Undefined metric denominators also produce NaN.')


## 6. Visualize matched test examples
The first test tiles are selected by CSV order, independent of model performance. This avoids choosing only attractive results. Seed is fixed to the first configured seed, not the best test-performing seed. Orange marks false positives; magenta marks false negatives; light gray is ignored. For a deeper error analysis, also examine the per-event CSV and clearly label any deliberately selected failure examples.


In [ ]:
N_EXAMPLES = min(6,len(splits['test']))
fig,axes = plt.subplots(N_EXAMPLES,2+len(EXPERIMENTS),figsize=(4*(2+len(EXPERIMENTS)),3*N_EXAMPLES),squeeze=False)
models = {name:load_model(name,channels,SEEDS[0]) for name,channels in EXPERIMENTS.items()}
for row,pair in enumerate(splits['test'][:N_EXAMPLES]):
    x,y = read_pair(pair)
    axes[row,0].imshow(x[0],cmap='gray',vmin=-30,vmax=0)
    axes[row,0].set_title(f'{pair[0]} | VV')
    truth = np.zeros((*y.shape,3),dtype=np.float32)
    truth[y == 1] = [0.1,0.5,1.0]
    truth[y == IGNORE] = [0.8,0.8,0.8]
    axes[row,1].imshow(truth)
    axes[row,1].set_title('Ground truth: blue = water')
    for col,(name,channels) in enumerate(EXPERIMENTS.items(),2):
        z = ((x-MEAN[:,None,None])/STD[:,None,None])[channels]
        with torch.no_grad():
            logits = models[name](torch.from_numpy(z.copy()).float().unsqueeze(0).to(DEVICE))
        pred = logits[0].cpu().numpy() >= 0
        valid = y != IGNORE
        rgb = np.zeros((*y.shape,3),dtype=np.float32)
        rgb[pred & (y == 1) & valid] = [0.1,0.5,1.0]
        rgb[pred & (y == 0) & valid] = [1.0,0.5,0.0]
        rgb[~pred & (y == 1) & valid] = [1.0,0.0,0.8]
        rgb[~valid] = [0.8,0.8,0.8]
        axes[row,col].imshow(rgb)
        axes[row,col].set_title(f'{name}: blue TP, orange FP, magenta FN')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
fig.savefig(ROOT/'test_examples.png',dpi=150,bbox_inches='tight')
plt.show()
del models
if AMP:
    torch.cuda.empty_cache()


## 7. Export and report
The archive excludes the source imagery and contains checkpoints, CSV splits, data inventory, settings, software versions, results, plots, and a report outline. Download it before the runtime ends. Also save this notebook separately using the notebook interface.

Suggested report: question → dataset/labels → fixed experiment protocol → quantitative comparison → error examples → limitations. Describe results as observed; do not imply a publication, new architecture, or unseen-event evaluation. With one seed and one dataset, avoid broad claims that one channel configuration is universally superior.


In [ ]:
environment = dict(python=platform.python_version(),torch=torch.__version__,
                   numpy=np.__version__,rasterio=rasterio.__version__,pandas=pd.__version__,
                   device=str(DEVICE),gpu=torch.cuda.get_device_name(0) if AMP else None)
(ROOT/'environment.json').write_text(json.dumps(environment,indent=2))
report = """# Sentinel-1 channel comparison: pilot study

## Question
Does VV + VH improve surface-water segmentation compared with either channel alone?

## Protocol
Official Sen1Floods11 standard splits; hand-labelled water/non-water targets.
Small U-Net; same budget, loss and augmentation; training-only normalization.
Best validation-IoU checkpoint; fixed 0.5 threshold. Consult runs/*/config.json.

## Results
Insert test_results.csv and training_curves.png. If multiple seeds were used,
report mean and sample standard deviation. Do not substitute estimated scores.

## Error analysis
Discuss test_examples.png and event-specific results. Explain observed errors
without asserting an unverified physical cause.

## Limitations
Short training budget; small dataset; finite seed count; no independent external
test event study. Water labels do not distinguish permanent water from flooding.
Standard split performance does not establish unseen-event generalization.

## Attribution
Bonafilia, Tellman, Anderson, and Issenberg. Sen1Floods11: A Georeferenced Dataset
to Train and Test Deep Learning Flood Algorithms for Sentinel-1. CVPRW 2020.
https://github.com/cloudtostreet/Sen1Floods11
Data-loading approach informed by the authors' example notebook; this notebook
uses a separate small U-Net experiment pipeline. Review all generated code and
report your own executed results.
"""
(ROOT/'REPORT_OUTLINE.md').write_text(report)
export_dir = ROOT/'export'
export_dir.mkdir(exist_ok=True)
for path in ROOT.iterdir():
    if path.is_file() and path.suffix in ['.csv','.json','.png','.md']:
        shutil.copy2(path,export_dir/path.name)
shutil.copytree(RUNS,export_dir/'runs',dirs_exist_ok=True)
shutil.copytree(DATA/'splits',export_dir/'splits',dirs_exist_ok=True)
archive = shutil.make_archive(str(ROOT/'sen1_study_results'),'zip',export_dir)
print('Results archive:',archive)
from IPython.display import FileLink
display(FileLink(archive))
# Colab alternative (uncomment if needed):
# from google.colab import files
# files.download(archive)
